# LangChain: Evaluation

Building a Q&A system is one thing — but how do you know it actually works well?  
This notebook walks through a practical evaluation workflow:

1. **Build** the Q&A chain (LCEL, modern API)
2. **Create test examples** — hand-crafted and LLM-generated
3. **Run & inspect** queries manually
4. **Batch evaluate** answers with an LLM judge
5. **Interpret** the grading results

---
> **LangChain versions used:** `langchain>=1.3`, `langchain-openai>=1.4`, `langchain-community>=0.4`  
> All chains are written with **LCEL** (the pipe `|` syntax). The old `RetrievalQA` / `apply_and_parse` / `langchain.evaluation` APIs are fully replaced.

## Setup

In [1]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [2]:
LLM_MODEL = "gpt-3.5-turbo"

---
## Part 1 — Build the Q&A Chain

We load a product catalogue CSV into an **in-memory vector store**, then expose it as a retrieval-augmented Q&A chain using LCEL.

```
question ──► retriever (top-k docs) ──► prompt ──► LLM ──► answer
```

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

/var/folders/l3/6dn78ndx5cb9nbc_4r6nycq40000gn/T/ipykernel_86304/3176700477.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [4]:
loader = CSVLoader(file_path="assets/OutdoorClothingCatalog_1000.csv")
docs = loader.load()
print(type(docs))
print(f"Loaded {len(docs)} documents")
docs[0]

<class 'list'>
Loaded 1000 documents


Document(metadata={'source': 'assets/OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [5]:
# Build vector store from all documents.
# DocArrayInMemorySearch keeps everything in RAM — fine for small datasets.
embeddings = OpenAIEmbeddings()
vectorstore = DocArrayInMemorySearch.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [6]:
llm = ChatOpenAI(temperature=0.0, model=LLM_MODEL)

# The prompt passes retrieved catalogue snippets as context.
prompt = ChatPromptTemplate.from_template("""\
You are a helpful outdoor clothing assistant.
Answer the question using ONLY the product information provided below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}
""")

def format_docs(retrieved_docs):
    """Join retrieved document texts separated by a clear divider."""
    return "\n\n---\n\n".join(d.page_content for d in retrieved_docs)

# LCEL chain: retrieve → format → prompt → LLM → parse
qa_chain = (
    RunnableParallel(
        context=retriever | format_docs,
        question=RunnablePassthrough()
    )
    | prompt
    | llm
    | StrOutputParser()
)

print("Q&A chain ready.")

Q&A chain ready.


---
## Part 2 — Create Test Examples

Good evaluation needs representative questions with known correct answers.  
We'll create them two ways:
- **Hand-crafted**: written by a human, high confidence
- **LLM-generated**: the model reads each document and proposes a question/answer pair

### 2a. Inspect the Raw Documents

Read a couple of documents first so hand-crafted questions are grounded in real content.

In [7]:
docs[10]

Document(metadata={'source': 'assets/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [8]:
docs[11]

Document(metadata={'source': 'assets/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

### 2b. Hand-Crafted Examples

In [9]:
examples = [
    {
        "question": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "question": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    },
]

### 2c. LLM-Generated Examples

We ask the LLM to read each document and generate a question/answer pair.  
This is cheaper than writing many examples by hand, and tends to cover a wider range of the content.

The generation prompt instructs the model to return structured JSON: `{"query": "...", "answer": "..."}`

In [10]:
import json
from langchain_core.prompts import ChatPromptTemplate

qa_generation_prompt = ChatPromptTemplate.from_template("""\
You are an expert at generating evaluation examples for Q&A systems.

Given the product description below, generate ONE question that can be answered
using ONLY the information in the description, and provide the correct answer.

Return your response as valid JSON in exactly this format:
{{"query": "<the question>", "answer": "<the answer>"}}

Product description:
{doc}
""")

gen_llm = ChatOpenAI(temperature=0.0, model=LLM_MODEL)

qa_gen_chain = qa_generation_prompt | gen_llm | StrOutputParser()

print("Generation chain ready.")

Generation chain ready.


In [11]:
# Generate one Q&A pair per document for the first 5 docs.
# Each invocation is independent so we can call them in a simple loop.
raw_generated = qa_gen_chain.batch(
    [{"doc": doc.page_content} for doc in docs[:5]]
)

new_examples = []
for raw in raw_generated:
    try:
        new_examples.append(json.loads(raw))
    except json.JSONDecodeError:
        print(f"Skipping unparseable output: {raw[:80]}...")

print(f"Generated {len(new_examples)} examples")

Generated 5 examples


In [12]:
# Inspect the first generated example alongside its source document.
print("Generated example:")
print(json.dumps(new_examples[0], indent=2))
print()
print("Source document:")
print(docs[0].page_content)

Generated example:
{
  "query": "What is the approximate weight of one pair of Women's Campside Oxfords?",
  "answer": "1 lb. 1 oz. per pair"
}

Source document:
: 0
name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. 

Size & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. 

Specs: Approx. weight: 1 lb.1 oz. per pair. 

Construction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. 

Questions? Please contact us for any inquiries.


### 2d. Combine All Examples

Merge hand-crafted and generated examples into a single list.  
All items must have `"question"` and `"answer"` keys.

In [13]:
# Normalise generated examples: the generation prompt uses "query" — rename to "question"
for ex in new_examples:
    if "query" in ex and "question" not in ex:
        ex["question"] = ex.pop("query")

examples += new_examples
print(f"Total examples: {len(examples)}")

Total examples: 7


---
## Part 3 — Manual Evaluation

Before running automated grading, run a single question through the chain manually.  
This is the fastest way to spot obvious retrieval or generation problems.

**Debug mode** — setting `langchain.debug = True` prints every chain step in detail:  
which documents were retrieved, what the final prompt looked like, the raw LLM response, etc.

In [14]:
# Quick sanity check — run example 0 without debug output
answer = qa_chain.invoke(examples[0]["question"])
print(f"Q: {examples[0]['question']}")
print(f"A: {answer}")

Q: Do the Cozy Comfort Pullover Set have side pockets?
A: I don't know.


In [15]:
import langchain

langchain.debug = True  # verbose output for every chain step

In [16]:
qa_chain.invoke(examples[0]["question"])

"I don't know."

In [17]:
langchain.debug = False  # turn debug output off again

---
## Part 4 — LLM-Assisted Batch Evaluation

Manual inspection doesn't scale. Instead, we use the LLM itself as a judge.

**Workflow:**
1. **Collect predictions** — run every example through `qa_chain`
2. **Grade predictions** — for each `(question, reference_answer, predicted_answer)`, ask the LLM whether the prediction is `CORRECT` or `INCORRECT`
3. **Summarise** results

> **Tip:** LLM evaluation is fast and cheap at this scale, but is itself imperfect — always sanity-check a few grades manually.

### 4a. Collect Predictions

In [18]:
questions = [ex["question"] for ex in examples]

# batch() is more efficient than calling invoke() in a loop.
predicted_answers = qa_chain.batch(questions)

print(f"Collected {len(predicted_answers)} predictions")

Collected 7 predictions


### 4b. Build the Grading Chain

The grader receives the question, the reference answer, and the model's answer.  
It returns a single word: `CORRECT` or `INCORRECT`.

In [19]:
eval_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an expert grader for a Q&A system.
You will be given a question, the correct reference answer, and a predicted answer.
Grade the predicted answer as CORRECT if it captures the essential meaning of the reference answer,
even if the wording differs. Otherwise grade it INCORRECT.
Respond with ONLY one word: CORRECT or INCORRECT."""),
    ("human",
     """Question: {question}
Reference answer: {reference_answer}
Predicted answer: {predicted_answer}

Grade:"""),
])

eval_llm = ChatOpenAI(temperature=0, model=LLM_MODEL)
eval_chain = eval_prompt | eval_llm | StrOutputParser()

print("Evaluation chain ready.")

Evaluation chain ready.


### 4c. Grade All Predictions

In [20]:
eval_inputs = [
    {
        "question": ex["question"],
        "reference_answer": ex["answer"],
        "predicted_answer": pred,
    }
    for ex, pred in zip(examples, predicted_answers)
]

grades = eval_chain.batch(eval_inputs)
print(f"Graded {len(grades)} predictions")

Graded 7 predictions


### 4d. Results

In [21]:
for i, (ex, pred, grade) in enumerate(zip(examples, predicted_answers, grades)):
    grade_label = grade.strip().upper()
    icon = "✅" if grade_label == "CORRECT" else "❌"
    print(f"Example {i}  {icon}  {grade_label}")
    print(f"  Q:         {ex['question']}")
    print(f"  Expected:  {ex['answer']}")
    print(f"  Predicted: {pred}")
    print()

Example 0  ❌  INCORRECT
  Q:         Do the Cozy Comfort Pullover Set have side pockets?
  Expected:  Yes
  Predicted: I don't know.

Example 1  ✅  CORRECT
  Q:         What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
  Expected:  The DownTek collection
  Predicted: DownTek collection

Example 2  ✅  CORRECT
  Q:         What is the approximate weight of one pair of Women's Campside Oxfords?
  Expected:  1 lb. 1 oz. per pair
  Predicted: Approx. weight: 1 lb.1 oz. per pair.

Example 3  ✅  CORRECT
  Q:         What are the dimensions of the medium size Recycled Waterhog Dog Mat, Chevron Weave?
  Expected:  Dimensions: 22.5" x 34.5"
  Predicted: Dimensions: 22.5" x 34.5"

Example 4  ✅  CORRECT
  Q:         What is the sun protection rating of the fabric used in the swimsuit?
  Expected:  UPF 50+
  Predicted: UPF 50+ rated - the highest rated sun protection possible.

Example 5  ✅  CORRECT
  Q:         What is the sun protection rating of the Refresh Swimwear, V-Neck

### 4e. Summary Score

In [22]:
total = len(grades)
correct = sum(1 for g in grades if g.strip().upper() == "CORRECT")

print(f"Score: {correct}/{total}  ({correct / total:.0%} accuracy)")

Score: 6/7  (86% accuracy)


---
## Summary

| Step | What we did |
|------|-------------|
| Build | LCEL retrieval chain: `retriever → prompt → LLM → parser` |
| Test data | 2 hand-crafted + 5 LLM-generated Q&A pairs |
| Manual eval | `langchain.debug = True` to inspect a full chain run |
| Auto eval | LLM judge grades each prediction as CORRECT / INCORRECT |

**Next steps:**
- Tune `k` (number of retrieved docs) and see how accuracy changes
- Try different embedding models or chunk sizes
- For large-scale evaluation, look into **LangSmith** — it provides a hosted eval platform with trace-level visibility